# Chapter 6 — Order Changes Meaning

## Question

**If two bundles contain the same information but arrange it differently, are they the same context?**

Structural answer: no. Membership fingerprints match while rendered fingerprints differ, and measurable properties — position, adjacency, grouping, temporal sequence, stable-prefix length — move. No ordering is claimed behaviourally superior: no model runs here.

## Setup — seven fixed items with authority metadata

Each item carries `authority` and `scope` as separate variables from its slot. Moving an item changes position; the metadata does not move with it.

In [ ]:
import hashlib
from dataclasses import dataclass

@dataclass
class BundleItem:
    id: str
    label: str
    text: str
    tokens: int
    authority: str
    scope: str

ITEMS = {
    'sys':     BundleItem('sys', 'System instruction', 'Act as a careful coding agent.', 60, 'vendor', 'global'),
    'rules':   BundleItem('rules', 'Project constraints', 'Never modify migration files. Approval required.', 80, 'project', 'project'),
    'tools':   BundleItem('tools', 'Tool definition', 'TOOL run_tests(target).', 120, 'descriptive', 'task'),
    'state':   BundleItem('state', 'Task state', 'Goal: fix checkout total. Tried: updated formula.', 90, 'derived', 'task'),
    'evidence': BundleItem('evidence', 'Evidence', 'Failing assertion: expected 42.00, got 41.50.', 70, 'evidence', 'project'),
    'recent':  BundleItem('recent', 'Recent observation', 'Lint clean; focused test still red.', 60, 'data', 'observation'),
    'request': BundleItem('request', 'Current request', 'Run the focused test again.', 40, 'user', 'turn'),
}

def sha(b: str) -> str:
    return hashlib.sha256(b.encode('utf-8')).hexdigest()[:16]

def membership_fingerprint(order):
    return sha('|'.join(sorted(order)))

def render(order):
    return '\n\n'.join(f"[{ITEMS[i].label}] {ITEMS[i].text}" for i in order)

def rendered_fingerprint(order):
    return sha(render(order))

def norm_positions(order):
    total = sum(ITEMS[i].tokens for i in order)
    cum = 0
    out = {}
    for i in order:
        out[i] = (cum + ITEMS[i].tokens / 2) / total
        cum += ITEMS[i].tokens
    return out

BASE = ['sys', 'rules', 'tools', 'state', 'evidence', 'recent', 'request']
print(f'{len(ITEMS)} items, {sum(ITEMS[i].tokens for i in BASE)} fixture tokens.')

## Baseline — same membership, different rendered object

In [ ]:
REVERSED = list(reversed(BASE))
print('membership equal:', membership_fingerprint(BASE) == membership_fingerprint(REVERSED))
print('rendered equal:  ', rendered_fingerprint(BASE) == rendered_fingerprint(REVERSED))
assert membership_fingerprint(BASE) == membership_fingerprint(REVERSED)
assert rendered_fingerprint(BASE) != rendered_fingerprint(REVERSED)
print('A context bundle is a sequence, not a set.')

## Position — each item's normalised slot

In [ ]:
for i in BASE:
    print(f"{ITEMS[i].label:20s} base {norm_positions(BASE)[i]:5.1%}  reversed {norm_positions(REVERSED)[i]:5.1%}")

## Adjacency — claim with evidence, together versus split

Adjacent: the failing assertion sits directly under the task state that cites it. Separated: matched filler of equal token size divides them. Totals are matched; only distance changes.

In [ ]:
FILLER = BundleItem('filler', 'Filler', 'Unrelated project notes. ' * 20, 160, 'none', 'none')

def token_distance(order, extra, a, b):
    toks = {i: ITEMS[i].tokens for i in order if i in ITEMS}
    toks.update(extra)
    names = [x for x in order]
    between = names[names.index(a) + 1:names.index(b)]
    return sum(toks[x] for x in between)

adjacent = ['sys', 'rules', 'tools', 'state', 'evidence', 'recent', 'request']
separated = ['sys', 'rules', 'tools', 'state', 'filler_a', 'filler_b', 'evidence', 'recent', 'request']
extra = {'filler_a': 80, 'filler_b': 80}
assert set(separated) - {'filler_a', 'filler_b'} == set(adjacent)
d_adj = token_distance(adjacent, {}, 'state', 'evidence')
d_sep = token_distance(separated, extra, 'state', 'evidence')
print(f'claim-to-evidence gap adjacent: {d_adj} tokens; separated: {d_sep} tokens (matched filler: {sum(extra.values())})')
assert d_adj == 0 and d_sep == 160
print('No claim about which the model uses better: distance is structural, usability is behavioural.')

## Grouping — one block of rules versus dispersed rules

In [ ]:
RULES = ['never modify migration files', 'approval required for deploys', 'tests must pass before merge', 'no force-push to main']

def fragments(order):
    """Count maximal runs of rule spans: 1 means one consultation object."""
    runs, in_run = 0, False
    for x in order:
        if x == 'rule':
            if not in_run:
                runs += 1
                in_run = True
        else:
            in_run = False
    return runs

blocked = ['sys', 'rule', 'rule', 'rule', 'rule', 'tools', 'request']
dispersed = ['sys', 'rule', 'tools', 'rule', 'state', 'rule', 'evidence', 'rule', 'request']
assert sorted(x for x in blocked if x == 'rule') == sorted(x for x in dispersed if x == 'rule')
print(f'blocked fragments: {fragments(blocked)}; dispersed fragments: {fragments(dispersed)} (same four rules)')
assert fragments(blocked) == 1 and fragments(dispersed) == 4

## Temporal order — the same events, three sequences

In [ ]:
EVENTS = ['t1: reproduced failure', 't2: searched for total formula', 't3: edited formula', 't4: test still red']
chrono = list(EVENTS)
reverse = list(reversed(EVENTS))
role_grouped = ['t2: searched for total formula', 't3: edited formula', 't1: reproduced failure', 't4: test still red']
for name, seq in [('chronological', chrono), ('reverse-chronological', reverse), ('role-grouped', role_grouped)]:
    print(f'{name:22s} first: {seq[0]}')
assert sorted(chrono) == sorted(reverse) == sorted(role_grouped)
assert chrono != reverse
print('Same events, different sequences. No layout is endorsed.')

## Stable versus dynamic placement — prefix length is structural

Turn N+1 appends a fresh observation. Layout S keeps stable material early; layout V puts the volatile observation first. Byte-identical leading prefix is counted, not priced: cache savings belong to Chapter 9.

In [ ]:
def stable_prefix_len(prev_texts, next_texts):
    n = 0
    for a, b in zip(prev_texts, next_texts):
        if a == b:
            n += 1
        else:
            break
    return n

prev = [ITEMS[i].text for i in BASE]
layout_s = [ITEMS[i].text for i in BASE] + ['Fresh observation from turn N+1.']
layout_v = ['Fresh observation from turn N+1.'] + [ITEMS[i].text for i in BASE]
print(f'stable-first prefix items: {stable_prefix_len(prev, layout_s)}; volatile-first prefix items: {stable_prefix_len(prev, layout_v)}')
assert stable_prefix_len(prev, layout_s) == len(BASE)
assert stable_prefix_len(prev, layout_v) == 0

## Precedence — position cannot encode authority

Move the project rule and the user request through every slot. Authority metadata never changes; only slots do. Sequence and authority are separate variables; Chapter 19 owns conflicts.

In [ ]:
for order in (BASE, REVERSED):
    pos = norm_positions(order)
    print(f"rules at {pos['rules']:.0%}, request at {pos['request']:.0%}; rules authority still '{ITEMS['rules'].authority}', request still '{ITEMS['request'].authority}'")
assert ITEMS['rules'].authority == 'project' and ITEMS['request'].authority == 'user'
print('Recent does not mean authoritative; early does not mean important; loud does not mean governing.')

## One move, six effects — `project_constraints` above the request

In [ ]:
before = ['sys', 'rules', 'tools', 'state', 'evidence', 'recent', 'request']
after = ['sys', 'tools', 'state', 'evidence', 'recent', 'rules', 'request']
assert set(before) == set(after)
pb, pa = norm_positions(before), norm_positions(after)
print(f"position:            rules {pb['rules']:.0%} -> {pa['rules']:.0%}")
print(f"adjacency:           rules neighbour {[before[before.index('rules') + 1]]} -> {[after[after.index('rules') + 1]]}")
print(f"grouping:            rules leave the stable block (block split: sys|rules -> sys alone)")
print(f"temporal relation:   rules now follow evidence and recent (recency-adjacent to request)")
print(f"stable prefix (N+1 appends at end): shortened if rules counted as stable \u2014 layout-dependent")
print('unchanged:           item identity, authority (project), scope (project), membership')
assert ITEMS['rules'].authority == 'project' and ITEMS['rules'].scope == 'project'

## Observation — a hypothesised layout, labelled as such

HYPOTHESISED LAYOUT — NOT A UNIVERSAL RULE. A repeatable candidate for production, where each line must earn its slot in a one-variable fixture.

In [ ]:
CANONICAL = ['sys', 'rules', 'tools', 'state', 'evidence', 'recent', 'request']
print('HYPOTHESISED LAYOUT \u2014 NOT A UNIVERSAL RULE')
for i in CANONICAL:
    print(f'  {ITEMS[i].label} (authority: {ITEMS[i].authority})')
assert set(CANONICAL) == set(BASE)
assert rendered_fingerprint(CANONICAL) == rendered_fingerprint(BASE), 'the sketch reuses the baseline order; its status is hypothesis, not finding'

## Try it

1. Swap `evidence` and `recent` in `before` and list which measurements move while membership holds.
2. Change the filler sizes in the adjacency cell unequally and confirm the distance arithmetic still reconciles.
3. Disperse only two of the four rules and count fragments — grouping degrades by degrees, not by verdict.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# alt = ['sys', 'rules', 'tools', 'state', 'recent', 'evidence', 'request']
# print(membership_fingerprint(alt) == membership_fingerprint(BASE), rendered_fingerprint(alt) == rendered_fingerprint(BASE))

## What this demonstrates

- Context is ordered: identical membership renders measurably different bundles.
- Position, adjacency, grouping, temporal order, and stable-prefix structure each move independently and are each countable.
- Salience and authority are separate variables: slots change, metadata does not.
- One move changes many things at once, so experiments must log all six effects.

## What this does not demonstrate

- That important material belongs first, last, or anywhere in particular.
- That middle positions are unusable, or that adjacency/grouping always help behaviour.
- That the hypothesised canonical layout is optimal.
- That stable-prefix structure produces cache savings (Chapter 9 owns caching).

## Connection to the chapter

Ordering tells us where information appears. Under pressure we still need to know what kind of information each item is, and what transformations it is allowed to survive:

> Ordering tells us where information appears. Under pressure we still need to know what kind of information it is and what transformations it is allowed to survive.

That is Chapter 7.